# DIESEL — Notebook 04: TGDA Augmentation + Round 2 Training (Novel)
Applies Taxonomy-Guided Data Augmentation based on error analysis,
then runs Round 2 fine-tuning on the augmented dataset.

**Runtime:** Select the **L4 GPU** kernel.

**Prerequisites:** Notebooks 02 and 03 completed.

In [ ]:
!pip install -q torch transformers peft trl bitsandbytes
!pip install -q accelerate datasets sqlparse sqlglot
!pip install -q wandb matplotlib seaborn scipy scikit-learn python-dotenv tqdm

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# === Colab Setup: Clone project & set path ===
import os, sys

if not os.path.exists('/content/Text-to-SQL-LLM/src'):
    !git clone https://github.com/Deepak-Lingala/Text-to-SQL-LLM.git /content/Text-to-SQL-LLM

os.chdir('/content/Text-to-SQL-LLM')
sys.path.insert(0, '/content/Text-to-SQL-LLM')
print('Project root:', os.getcwd())

import torch, gc

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
ERROR_ANALYSIS_PATH = 'outputs/error_analysis/error_analysis_round1_finetuned.json'
ROUND1_ADAPTER = 'outputs/round1/final_adapter'
DRY_RUN = True  # Set to False for full training

In [ ]:
from src.config import get_config
from dataclasses import replace

config = get_config()

if DRY_RUN:
    config = replace(config, training=replace(
        config.training,
        max_steps=10,
        logging_steps=1,
        eval_steps=10,
        save_steps=10,
        report_to='none',
    ))
    print('*** DRY RUN MODE ***')

## Step 1: Load Error Analysis

In [ ]:
print('Loading error analysis results...')
with open(ERROR_ANALYSIS_PATH, 'r') as f:
    error_analysis = json.load(f)

print(f"Model: {error_analysis.get('model_name', 'unknown')}")
print(f"Accuracy: {error_analysis.get('overall_accuracy', 0):.1%}")
print(f"Errors classified: {error_analysis.get('total_incorrect', 0)}")

## Step 2: Load Spider Data

In [ ]:
from src.data_loader import load_spider_dataset, prepare_training_data

dataset, schema_manager = load_spider_dataset(config)
train_data = prepare_training_data(dataset, schema_manager, config, split='train')
eval_data = prepare_training_data(dataset, schema_manager, config, split='validation')
print(f'Train: {len(train_data)}, Eval: {len(eval_data)}')

## Step 3: Run TGDA

In [ ]:
from src.augmentor import run_tgda

train_list = [train_data[i] for i in range(len(train_data))]

augmented_dataset = run_tgda(
    error_analysis=error_analysis,
    original_train_data=train_list,
    eval_predictions=error_analysis.get('classified_errors', []),
    schema_manager=schema_manager,
    config=config,
)

print(f'\nAugmented dataset: {len(augmented_dataset)} examples')

## Step 4: Round 2 Training

In [ ]:
from src.train import train_round2

trainer, result = train_round2(
    augmented_dataset=augmented_dataset,
    eval_dataset=eval_data,
    config=config,
    round1_adapter_path=ROUND1_ADAPTER,
)

In [ ]:
del trainer; gc.collect(); torch.cuda.empty_cache()

print('\n' + '=' * 60)
print('Round 2 Training Complete!')
print('=' * 60)
print(f"Adapter saved to: {os.path.join(config.paths.round2_dir, 'final_adapter')}")
print(f"Loss: {result.metrics.get('train_loss', 'N/A')}")
print(f"Runtime: {result.metrics.get('train_runtime', 0):.0f}s")
print(f'\nNext: Run 05_final_evaluation.ipynb')